In [1]:
import os

# Configurar el nuevo token de la API de Kaggle como variable de entorno
os.environ['KAGGLE_API_TOKEN'] = "KGAT_e15a85fb6493778262368e6eaba23952"

print("¡Credenciales configuradas con éxito!")

¡Credenciales configuradas con éxito!


In [2]:
# Descargar el dataset de Kaggle
!kaggle datasets download -d uciml/pima-indians-diabetes-database

# Descomprimir el archivo descargado
!unzip -o pima-indians-diabetes-database.zip

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
unzip:  cannot find or open pima-indians-diabetes-database.zip, pima-indians-diabetes-database.zip.zip or pima-indians-diabetes-database.zip.ZIP.


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# --- 1. ESTUDIO DEL DATASET ---
# MODIFICACIÓN: Cargamos el dataset directamente desde internet para evitar el error de Kaggle
df = pd.read_csv('https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv')
print("--- Primeras 5 filas del dataset ---")
display(df.head()) # 'display' se ve mucho mejor en Colab que 'print'

# Características (Features) y Rótulo (Target)
X = df.drop('Outcome', axis=1) # Todas las columnas menos el resultado
y = df['Outcome']              # 1: Diabetes, 0: Sin Diabetes

print(f"\nCaracterísticas (Features): {list(X.columns)}")
print(f"Rótulo a predecir: 'Outcome'")

# Normalización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# --- 2. GENERAR EL MODELO ---
model = Sequential([
    Dense(16, input_dim=8, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid') # Sigmoid para salidas binarias (0 o 1)
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Entrenar (usamos verbose=1 para que veas el progreso en Colab)
print("\n--- Entrenando la Red Neuronal ---")
historial = model.fit(X_train, y_train, epochs=50, batch_size=10, validation_split=0.2, verbose=0)

# Evaluar
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Precisión del modelo en datos de prueba: {accuracy*100:.2f}%")

--- Primeras 5 filas del dataset ---


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



Características (Features): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Rótulo a predecir: 'Outcome'

--- Entrenando la Red Neuronal ---


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



✅ Precisión del modelo en datos de prueba: 72.08%


In [4]:
# --- 3. EJEMPLOS DE INTERNET Y ANÁLISIS ---
# Formato: [Embarazos, Glucosa, PresiónSangre, GrosorPiel, Insulina, IMC, FunciónPedigrí, Edad]

# Ejemplo A: Perfil sano encontrado en un foro de salud
# Ejemplo B: Perfil con indicadores altos de riesgo de una publicación médica
ejemplos_internet = np.array([
    [1, 85, 66, 29, 0, 26.6, 0.351, 31],   # Ejemplo A
    [5, 166, 72, 19, 175, 25.8, 0.587, 51] # Ejemplo B
])

# Importante: Escalar los datos nuevos
ejemplos_escalados = scaler.transform(ejemplos_internet)
predicciones = model.predict(ejemplos_escalados)

print("\n--- Análisis de Resultados ---")
for i, prediccion in enumerate(predicciones):
    probabilidad = prediccion[0]
    resultado = "POSITIVO para Diabetes" if probabilidad >= 0.5 else "NEGATIVO para Diabetes"
    print(f"Ejemplo {i+1}:")
    print(f"  - Probabilidad de diabetes: {probabilidad*100:.1f}%")
    print(f"  - Predicción de la red: {resultado}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

--- Análisis de Resultados ---
Ejemplo 1:
  - Probabilidad de diabetes: 3.0%
  - Predicción de la red: NEGATIVO para Diabetes

Ejemplo 2:
  - Probabilidad de diabetes: 68.4%
  - Predicción de la red: POSITIVO para Diabetes



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
